# Zone-Based Time-Series Measurements with Numerical Values and Identifiers Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the Zone-Based Time-Series Measurements with Numerical Values and Identifiers dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.wd1r-6eqn/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset name and description
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and their `@id`s.

In [ ]:
# List all available record sets in the dataset, with their @id and fields
record_sets = dataset.record_sets
if len(record_sets) == 0:
    print("No RecordSets found in the Croissant metadata. Please check the definition.")
else:
    for record_set in record_sets:
        print(f"RecordSet @id: {record_set['@id']}, name: {record_set.get('name', '')}")
        if 'field' in record_set:
            print("  Fields (@id):")
            for field in record_set['field']:
                if isinstance(field, dict):
                    print(f"    {field.get('@id', '')}  (name: {field.get('name','')})")
                else:
                    print(f"    {field}")
        print()

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Here, we dynamically extract all record sets into DataFrames
dataframes = {}
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
if len(record_set_ids) == 0:
    print("No record sets available to extract.")
else:
    for record_set_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            if len(records) > 0:
                df = pd.DataFrame(records)
                dataframes[record_set_id] = df
                print(f"Loaded RecordSet @id: {record_set_id}")
                print("Columns:", df.columns.tolist())
                print(df.head(2))
            else:
                print(f"No records found for RecordSet @id: {record_set_id}")
        except Exception as e:
            print(f"Failed to load RecordSet {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping or aggregating the data.

**Note:** You may need to adapt field/column names or choose appropriate fields based on the record set content.

In [ ]:
# Choose a record set for in-depth analysis (the first one, if available)
if len(dataframes) == 0:
    print("No DataFrames were extracted to analyze.")
else:
    # Pick the first available record set
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id].copy()
    print(f"Analyzing RecordSet: {record_set_id}. Columns: {df.columns.tolist()}")

    # Try to identify a numeric field
    numeric_candidates = df.select_dtypes(include=[float, int]).columns
    if len(numeric_candidates) == 0:
        # Try to coerce any field to numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except:
                continue
        numeric_candidates = df.select_dtypes(include=[float, int]).columns

    if len(numeric_candidates) == 0:
        print("No numeric fields found for EDA.")
    else:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean()

        # Basic filtering
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Records where {numeric_field} > {threshold} (mean): {len(filtered_df)} rows")
        print(filtered_df.head())

        # Normalization (Z-score)
        filtered_df[numeric_field + '_normalized'] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print("\nNormalized (z-score) values:")
        print(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

        # Try to groupby a likely 'session' or 'participant' column if it exists
        group_fields = [col for col in df.columns if any(key in col.lower() for key in ['participant', 'session', 'id'])]
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(name=f"mean_{numeric_field}")
            print(f"\nGrouped mean {numeric_field} by '{group_field}':")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions or field-to-field relationships using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt

if len(dataframes) > 0 and len(numeric_candidates) > 0:
    plt.figure(figsize=(8, 5))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If any other numeric columns exist, a scatterplot
    if len(numeric_candidates) > 1:
        plt.figure(figsize=(7, 5))
        plt.scatter(df[numeric_candidates[0]], df[numeric_candidates[1]], alpha=0.5)
        plt.xlabel(numeric_candidates[0])
        plt.ylabel(numeric_candidates[1])
        plt.title(f"Scatterplot: {numeric_candidates[0]} vs. {numeric_candidates[1]}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR^2 zone-based time-series dataset using the `mlcroissant` library. We performed an overview of available record sets, extracted tabular data, performed basic statistics and normalization, and visualized distributions in the data.

For complete analysis, explore more record sets and fields based on your domain use case. All references to record sets, fields, and columns in this notebook utilized their Croissant schema `@id` to ensure reproducibility.